# 지출내역 프로그램(Console)

In [ ]:
import csv

def add_expense(expenses):
    date = input("날짜(YYYY-MM-DD): ").strip()
    category = input("카테고리: ").strip()
    description = input("내용: ").strip()

    if not date or not category or not description:
        print("날짜, 카테고리, 내용은 비워 둘 수 없습니다.")
        return
    
    try:
        amount = int(input("금액: "))
    except ValueError:
        print("금액은 정수로 입력해 주세요.")
        return
    
    if amount <= 0:
        print("금액은 0보다 큰 값으로 입력해 주세요.")
        return

    expense = {
        "date": date,
        "category": category,
        "description": description,
        "amount": amount,
    }

    expenses.append(expense)
    print("지출 내역을 추가했습니다.")

def show_expenses(expenses):
    if not expenses:
        print("등록된 지출이 없습니다.")
        return
    print("\n=== 지출 내역 ===")
    number = 1

    columns = expenses[0].keys()
    print(f"{' | '.join(columns)}")
    print("-" * 40)

    for expense in expenses:
        print(
            f"{number}. {expense['date']} | "
            f"{expense['category']} | "
            f"{expense['description']} | "
            f"{expense['amount']:,}원"
        )
        number += 1

def calculate_total(expenses):
    total = 0
    for expense in expenses:
        total += expense["amount"]
    return total

def calculate_by_category(expenses):
    category_totals = {}
    for expense in expenses:
        category = expense["category"]
        amount = expense["amount"]
        if category in category_totals:
            category_totals[category] += amount
        else:
            category_totals[category] = amount
    return category_totals

def save_expenses(file_path, expenses):
    fieldnames = ["date", "category", "description", "amount"]
    with open(file_path, "w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(expenses)

def load_expenses(file_path):
    expenses = []
    try:
        with open(file_path, "r", encoding="utf-8-sig", newline="") as file:
            reader = csv.DictReader(file)
            for row in reader:
                try:
                    row["amount"] = int(row["amount"])
                except ValueError:
                    print("금액이 올바르지 않은 행은 건너뜁니다:", row)
                    continue
                expenses.append(row)
    except FileNotFoundError:
        return []
    return expenses


expenses = load_expenses("sample_expenses.csv")

while True:
    print("\n=== 개인 지출 관리 ===")
    print("1. 지출 추가")
    print("2. 지출 목록")
    print("3. 지출 요약")
    print("4. 저장")
    print("0. 종료")

    choice = input("메뉴 선택: ").strip()

    if choice == "1":
        add_expense(expenses)
    elif choice == "2":
        show_expenses(expenses)
    elif choice == "3":
        print("전체 지출:", calculate_total(expenses))
        print(calculate_by_category(expenses))
    elif choice == "4":
        save_expenses("sample_expenses.csv", expenses)
        print("저장했습니다.")
    elif choice == "0":
        break
    else:
        print("메뉴 번호를 다시 선택해 주세요.")

# 지출 내역 프로그램(Web)

In [ ]:
import csv
import os
import pandas as pd
import streamlit as st

# 1. 페이지 설정 (가장 처음에 위치)
st.set_page_config(
    page_title="개인 지출 관리 웹 서비스", page_icon="💰", layout="wide"
)

# 2. 우측 상단 Deploy 버튼, 헤더, 푸터 및 기본 메뉴 숨기기 + 탭 스타일 반응형 CSS 주입
hide_streamlit_style = """
    <style>
    #MainMenu {visibility: hidden;}
    header {visibility: hidden;}
    footer {visibility: hidden;}
    .stDeployButton {display:none;}
    
    /* 탭(Tab) 스타일 커스텀: 글자 크기 확대 및 흰색 텍스트 적용 */
    .stTabs [data-baseweb="tab-list"] {
        gap: 15px;
        background-color: transparent;
    }
    .stTabs [data-baseweb="tab"] {
        height: 50px;
        white-space: pre-wrap;
        background-color: transparent;
        border-radius: 4px;
        color: #FFFFFF !important;
        font-size: 25px !important;
        font-weight: 500;
        padding: 0px 20px;
    }
    .stTabs [aria-selected="true"] {
        background-color: rgba(255, 255, 255, 0.1) !important;
        border-bottom: 2px solid #FFFFFF !important;
    }
    
    /* 모바일 환경 대응 */
    @media (max-width: 768px) {
        .stTabs [data-baseweb="tab"] {
            font-size: 20px !important;
            padding: 0px 10px;
        }
    }
    </style>
"""
st.markdown(hide_streamlit_style, unsafe_allow_html=True)

FILE_PATH = "sample_expenses.csv"


# 파일 로드 함수
def load_expenses(file_path):
  expenses = []
  if not os.path.exists(file_path):
    return expenses
  try:
    with open(file_path, "r", encoding="utf-8-sig", newline="") as file:
      reader = csv.DictReader(file)
      for row in reader:
        try:
          row["amount"] = int(row["amount"])
        except ValueError:
          continue
        expenses.append(row)
  except Exception:
    return []
  return expenses


# 파일 저장 함수
def save_expenses(file_path, expenses):
  fieldnames = ["date", "category", "description", "amount"]
  with open(file_path, "w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(expenses)


# 앱 타이틀
st.title("개인 지출 관리 웹 서비스")

# 데이터 로드 (세션 상태 유지)
if "expenses" not in st.session_state:
  st.session_state.expenses = load_expenses(FILE_PATH)

# 상단 탭 메뉴 구성 (모바일에서도 화면 상단에 고정되어 항상 보입니다)
tab1, tab2, tab3 = st.tabs(["지출 목록 보기", "지출 추가하기", "지출 요약"])

# 1. 지출 목록 보기 탭
with tab1:
  st.subheader("등록된 지출 내역")

  if not st.session_state.expenses:
    st.info("등록된 지출 내역이 없습니다.")
  else:
    df = pd.DataFrame(st.session_state.expenses)
    df["amount"] = df["amount"].apply(lambda x: f"{x:,}원")
    st.dataframe(df, use_container_width=True)

    if st.button("변경사항 파일로 저장"):
      save_expenses(FILE_PATH, st.session_state.expenses)
      st.success("파일에 성공적으로 저장되었습니다!")

# 2. 지출 추가하기 탭
with tab2:
  st.subheader("새로운 지출 내역 추가")

  with st.form("expense_form"):
    date = st.date_input("날짜")
    category = st.text_input("카테고리 (예: 식비, 교통비, 주거)")
    description = st.text_input("내용 (예: 점심값, 지하철)")
    amount = st.number_input("금액 (원)", min_value=0, step=1000)

    submitted = st.form_submit_button("지출 추가")

    if submitted:
      if not category.strip() or not description.strip():
        st.error("카테고리와 내용은 비워 둘 수 없습니다.")
      elif amount <= 0:
        st.error("금액은 0보다 큰 값이어야 합니다.")
      else:
        new_item = {
            "date": str(date),
            "category": category.strip(),
            "description": description.strip(),
            "amount": int(amount),
        }
        st.session_state.expenses.append(new_item)
        save_expenses(FILE_PATH, st.session_state.expenses)
        st.success("지출 내역이 추가되고 저장되었습니다!")

# 3. 지출 요약 탭
with tab3:
  st.subheader("지출 요약 및 통계")

  if not st.session_state.expenses:
    st.info("요약할 지출 내역이 없습니다.")
  else:
    df = pd.DataFrame(st.session_state.expenses)

    total_spent = df["amount"].sum()
    st.metric(label="총 지출 금액", value=f"{total_spent:,}원")

    st.markdown("---")
    st.markdown("### 카테고리별 지출 현황")

    category_df = df.groupby("category")["amount"].sum().reset_index()
    category_df["amount_formatted"] = category_df["amount"].apply(
        lambda x: f"{x:,}원"
    )

    st.table(category_df[["category", "amount_formatted"]])

# TABLE DDL

In [ ]:
CREATE TABLE expenses.expenses (
	id serial4 NOT NULL,
	"date" text NOT NULL,
	category text NOT NULL,
	description text NOT NULL,
	amount int4 NOT NULL,
	CONSTRAINT expenses_amount_not_null NOT NULL amount,
	CONSTRAINT expenses_category_not_null NOT NULL category,
	CONSTRAINT expenses_date_not_null NOT NULL date,
	CONSTRAINT expenses_description_not_null NOT NULL description,
	CONSTRAINT expenses_id_not_null NOT NULL id,
	CONSTRAINT expenses_pkey PRIMARY KEY (id)
);